# Multi-seed comparison (notebook) — AttentionKAN with LTN vs. without LTN

Notebook version of `run_multiseed.py`, split into cells for step-by-step
debugging. Each model type is trained for a FIXED number of epochs =
(its convergence epoch) x 1.10, looped over seeds; then we print a mean+/-std
table plus paired Wilcoxon (across seeds) and McNemar (per seed) tests.

Run order: Imports -> Config -> LTN setup -> Data -> Model/train fns ->
Run loop -> Summary -> Significance -> Save. You can re-run any cell in
isolation (e.g. inspect `tracker` after the run loop).


In [8]:
# --- Imports & device ---
import os, sys, json, time
import numpy as np
import pandas as pd
import torch

sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('../P1_structurelevel'))

from utils import LogitsToPredicate, MultiKANModel, DataLoader, DataLoaderMulti
from attention_modules import AttentionKANModel
from kan import KAN
from sklearn.preprocessing import StandardScaler
from eval_metrics import (set_seed, evaluate_run, MetricTracker,
                          wilcoxon_compare, mcnemar_across_seeds)
import ltn, ltn.fuzzy_ops

# Device: CUDA if available, otherwise CPU.
# NOTE: we deliberately do NOT use MPS. pykan and LTN both manage device
# placement themselves; on MPS the LTN Constants stay on CPU while the model is
# on the GPU, so the predicate hands a CPU tensor to an MPS model ->
# "two devices, mps:0 and cpu". MPS also doesn't speed up this small KAN. CPU is
# robust and about as fast here.
if torch.cuda.is_available():
    device = torch.device('cuda:0')
else:
    device = torch.device('cpu')
print('device:', device)


device: cpu


In [9]:
# --- Config (edit here) ---
TRAIN_PATH = '../P1_structurelevel/efficiency/input_files/logiKNet_train_10000.csv'
TEST_PATH  = '../P1_structurelevel/efficiency/input_files/logiKNet_test_3994.csv'

# run controls (replace the old argparse flags)
SEEDS = [0, 1, 2, 3, 4]
FAST  = False     # True -> train on the small 3994-row file (quick wiring check)

X_columns = [
    'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate',
    'IPv', 'LLC',
    'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number',
    'Magnitue', 'Radius', 'Covariance',
]
BENIGN_L2   = 5
IN_FEATURES = len(X_columns)        # 18
N_CLASSES   = 6
KAN_WIDTH   = [IN_FEATURES, 6, 6, N_CLASSES]
ATTN = dict(d_model=32, n_heads=4, n_layers=2, dropout=0.1, residual=True)
LR   = 1e-3

# convergence epochs measured from the sanity notebooks (EDIT IF SWAPPED)
CONVERGE = {"ltn": 625, "noltn": 843}
MARGIN   = 0.05
EPOCHS   = {k: int(round(v * (1 + MARGIN))) for k, v in CONVERGE.items()}

CHILD_TO_PARENT_2_6 = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 1}
PARTIAL = 0.5

SAVE_DIR = './saved'
os.makedirs(SAVE_DIR, exist_ok=True)
print('fixed epoch budgets (+%.0f%%):' % (MARGIN*100), EPOCHS)


fixed epoch budgets (+5%): {'ltn': 656, 'noltn': 885}


In [10]:
# --- LTN setup (6-class 2_6 hierarchy, matches AttentionEncoder.ipynb) ---
Not    = ltn.Connective(ltn.fuzzy_ops.NotStandard())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
SatAgg = ltn.fuzzy_ops.SatAgg()

l_c0 = ltn.Constant(torch.tensor([1, 0, 0, 0, 0, 0], device=device))   # MQTT-DDoS-Connect_Flood
l_c1 = ltn.Constant(torch.tensor([0, 1, 0, 0, 0, 0], device=device))   # MQTT-DDoS-Publish_Flood
l_c2 = ltn.Constant(torch.tensor([0, 0, 1, 0, 0, 0], device=device))   # MQTT-DoS-Connect_Flood
l_c3 = ltn.Constant(torch.tensor([0, 0, 0, 1, 0, 0], device=device))   # MQTT-DoS-Publish_Flood
l_c4 = ltn.Constant(torch.tensor([0, 0, 0, 0, 1, 0], device=device))   # MQTT-Malformed_Data
l_c5 = ltn.Constant(torch.tensor([0, 0, 0, 0, 0, 1], device=device))   # Benign

def compute_sat_levels(loader, P):
    sat_level = 0
    for data, labels in loader:
        x_c0 = ltn.Variable("x_c0", data[labels == 0])
        x_c1 = ltn.Variable("x_c1", data[labels == 1])
        x_c2 = ltn.Variable("x_c2", data[labels == 2])
        x_c3 = ltn.Variable("x_c3", data[labels == 3])
        x_c4 = ltn.Variable("x_c4", data[labels == 4])
        x_c5 = ltn.Variable("x_c5", data[labels == 5])
        x_MQTT = ltn.Variable("x_MQTT", data[labels < 5])
        sat_level = SatAgg(
            Forall(x_c0, P(x_c0, l_c0)),
            Forall(x_c1, P(x_c1, l_c1)),
            Forall(x_c2, P(x_c2, l_c2)),
            Forall(x_c3, P(x_c3, l_c3)),
            Forall(x_c4, P(x_c4, l_c4)),
            Forall(x_c5, P(x_c5, l_c5)),
            Forall(x_MQTT, Not(P(x_MQTT, l_c5))),   # MQTT is not Benign
        )
    return sat_level


In [11]:
# --- Data ---
train_df = pd.read_csv(TEST_PATH if FAST else TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
if FAST:
    print('[FAST] training on the 3994-row file (quick check only)')

for d in (train_df, test_df):
    d['label_L1'] = (d['label_L2'] == BENIGN_L2).astype(int)

scaler = StandardScaler()
Xtr = scaler.fit_transform(train_df[X_columns])
Xte = scaler.transform(test_df[X_columns])
ytr = train_df[['label_L1', 'label_L2']].values
yte = test_df[['label_L1', 'label_L2']].values
print('train shape:', Xtr.shape, '| test shape:', Xte.shape)
print('train L2 dist:', np.bincount(ytr[:, 1]))

# both loaders are 2-tuple (data, label_L2) -- this is what eval_metrics expects
train_loader = DataLoader(
    data=torch.tensor(Xtr, dtype=torch.float32, device=device),
    labels=torch.tensor(ytr[:, 1], dtype=torch.long, device=device),
    batch_size=len(train_df), shuffle=False)
# eval loader, fixed order (shuffle=False) so McNemar pairing is consistent
test_loader = DataLoader(
    data=torch.tensor(Xte, dtype=torch.float32, device=device),
    labels=torch.tensor(yte[:, 1], dtype=torch.long, device=device),
    batch_size=len(test_df), shuffle=False)


train shape: (10000, 18) | test shape: (3994, 18)
train L2 dist: [1688 1691 1700 1697 1542 1682]


In [12]:
# --- Model builder + training fns ---
def build_model(seed):
    kan = KAN(width=KAN_WIDTH, grid=5, k=3, seed=seed, device=device)
    return AttentionKANModel(IN_FEATURES, MultiKANModel(kan), **ATTN).to(device)

def train_ltn(model, train_loader, epochs):
    P = ltn.Predicate(LogitsToPredicate(model))
    opt = torch.optim.Adam(P.parameters(), lr=LR)
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        loss = 1. - compute_sat_levels(train_loader, P)
        loss.backward(); opt.step()
    return model

def train_noltn(model, train_loader, epochs):
    criterion = torch.nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        logits = model(train_loader.data, training=True)
        loss = criterion(logits, train_loader.labels)
        loss.backward(); opt.step()
    return model

MODELS = {
    "AttnKAN+LTN":    ("ltn",   train_ltn),
    "AttnKAN(noLTN)": ("noltn", train_noltn),
}


## Quick single-run debug (optional)

Run this before the full loop to confirm one model trains end-to-end and to
time it. Comment out / skip once you're happy.

In [13]:
# optional: one quick run to sanity-check + time before the full loop
set_seed(0)
_t = time.time()
_m = train_noltn(build_model(0), train_loader, 20)   # 20 epochs only
_r = evaluate_run(_m, test_loader, n_classes=N_CLASSES, device=device,
                  child_to_parent=CHILD_TO_PARENT_2_6, partial=PARTIAL)
print(f"debug run: acc {_r['accuracy']:.4f} | macroF1 {_r['macro_f1']:.4f} "
      f"| {time.time()-_t:.1f}s for 20 epochs")


checkpoint directory created: ./model
saving model version 0.0
debug run: acc 0.5128 | macroF1 0.4183 | 22.4s for 20 epochs


In [14]:
# --- Run loop: all models x all seeds (full budgets), saving each model ---
MODEL_DIR = os.path.join(SAVE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

def save_ckpt(model, key, seed, res):
    """Save one trained model's state_dict + everything needed to rebuild it."""
    path = os.path.join(MODEL_DIR, f"{key}_seed{seed}.pt")
    torch.save({
        "model_state": model.state_dict(),
        "model_type": key,          # 'ltn' or 'noltn'
        "seed": seed,
        "config": {"IN_FEATURES": IN_FEATURES, "N_CLASSES": N_CLASSES,
                   "KAN_WIDTH": KAN_WIDTH, "ATTN": ATTN,
                   "X_columns": X_columns, "BENIGN_L2": BENIGN_L2},
        "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_,
        "metrics": {k: v for k, v in res.items() if not k.startswith("_")},
    }, path)
    return path

tracker = MetricTracker()
for name, (key, train_fn) in MODELS.items():
    n_epochs = EPOCHS[key]
    for seed in SEEDS:
        set_seed(seed)
        t0 = time.time()
        model = build_model(seed)
        model = train_fn(model, train_loader, n_epochs)
        res = evaluate_run(model, test_loader, n_classes=N_CLASSES,
                           device=device, child_to_parent=CHILD_TO_PARENT_2_6,
                           partial=PARTIAL)
        tracker.add(name, seed, res)
        ckpt = save_ckpt(model, key, seed, res)
        print(f"[{name}] seed {seed} | epochs {n_epochs} "
              f"| acc {res['accuracy']:.4f} | macroF1 {res['macro_f1']:.4f} "
              f"| reliab {res['reliability']:.4f} | {time.time()-t0:.1f}s "
              f"| saved {os.path.basename(ckpt)}")
print('\ndone. models saved under', MODEL_DIR)


checkpoint directory created: ./model
saving model version 0.0
[AttnKAN+LTN] seed 0 | epochs 656 | acc 0.8140 | macroF1 0.8136 | reliab 0.8897 | 1363.9s | saved ltn_seed0.pt
checkpoint directory created: ./model
saving model version 0.0
[AttnKAN+LTN] seed 1 | epochs 656 | acc 0.7396 | macroF1 0.7301 | reliab 0.8423 | 1362.8s | saved ltn_seed1.pt
checkpoint directory created: ./model
saving model version 0.0
[AttnKAN+LTN] seed 2 | epochs 656 | acc 0.7626 | macroF1 0.7523 | reliab 0.8618 | 1367.1s | saved ltn_seed2.pt
checkpoint directory created: ./model
saving model version 0.0
[AttnKAN+LTN] seed 3 | epochs 656 | acc 0.8150 | macroF1 0.8126 | reliab 0.8887 | 1353.6s | saved ltn_seed3.pt
checkpoint directory created: ./model
saving model version 0.0
[AttnKAN+LTN] seed 4 | epochs 656 | acc 0.7964 | macroF1 0.7924 | reliab 0.8807 | 1319.4s | saved ltn_seed4.pt
checkpoint directory created: ./model
saving model version 0.0
[AttnKAN(noLTN)] seed 0 | epochs 885 | acc 0.8027 | macroF1 0.8009 

In [15]:
# --- Summary: mean +/- std over seeds ---
print(tracker.summary(metrics=["accuracy", "macro_f1", "macro_recall",
                               "macro_fpr", "macro_auroc",
                               "reliability", "hierarchical_f1"]))


model                         accuracy        macro_f1    macro_recall       macro_fpr     macro_auroc     reliability hierarchical_f1
--------------------------------------------------------------------------------------------------------------------------------------
AttnKAN+LTN              0.786+/-0.033   0.780+/-0.037   0.788+/-0.033   0.043+/-0.007   0.966+/-0.006   0.873+/-0.020   0.873+/-0.020
AttnKAN(noLTN)           0.803+/-0.011   0.802+/-0.011   0.804+/-0.011   0.039+/-0.002   0.969+/-0.004   0.882+/-0.008   0.882+/-0.008


In [16]:
# --- Significance: AttnKAN+LTN vs AttnKAN(noLTN) ---
a, b = "AttnKAN+LTN", "AttnKAN(noLTN)"
sig = {}
print(f"paired significance: {a} vs {b}\n")
for metric in ["accuracy", "macro_f1", "reliability"]:
    w = wilcoxon_compare(tracker, a, b, metric=metric)
    sig[f"wilcoxon_{metric}"] = w
    print(f"Wilcoxon {metric:14s}: "
          f"mean {w.get('mean_a', float('nan')):.4f} vs {w.get('mean_b', float('nan')):.4f} "
          f"| stat {w['statistic']:.3f} | p {w['p_value']:.4g}")

mc = mcnemar_across_seeds(tracker, a, b)
sig["mcnemar_per_seed"] = mc
print("\nMcNemar per seed (b01=A wrong/B right, b10=A right/B wrong):")
for r in mc:
    print(f"  seed {r['seed']}: b01 {r['b01']:4d} | b10 {r['b10']:4d} "
          f"| stat {r['statistic']:.3f} | p {r['p_value']:.4g}")


paired significance: AttnKAN+LTN vs AttnKAN(noLTN)

Wilcoxon accuracy      : mean 0.7855 vs 0.8030 | stat 6.000 | p 0.8125
Wilcoxon macro_f1      : mean 0.7802 vs 0.8020 | stat 6.000 | p 0.8125
Wilcoxon reliability   : mean 0.8726 vs 0.8816 | stat 6.000 | p 0.8125

McNemar per seed (b01=A wrong/B right, b10=A right/B wrong):
  seed 0: b01   93 | b10  138 | stat 8.381 | p 0.003792
  seed 1: b01  459 | b10  136 | stat 174.259 | p 8.691e-40
  seed 2: b01  330 | b10  171 | stat 49.828 | p 1.678e-12
  seed 3: b01   91 | b10  152 | stat 14.815 | p 0.0001186
  seed 4: b01  151 | b10  179 | stat 2.209 | p 0.1372


In [17]:
# --- Save results ---
out = {
    "epochs": {k: EPOCHS[v] for k, (v, _) in MODELS.items()},
    "seeds": SEEDS,
    "summary": {mdl: {m: tracker.mean_std(mdl, m)
                      for m in MetricTracker.SCALAR_KEYS
                      if tracker.store[mdl][m]}
                for mdl in tracker.store},
    "significance": sig,
}
res_path = os.path.join(SAVE_DIR, "multiseed_results.json")
with open(res_path, "w") as f:
    json.dump(out, f, indent=2, default=float)
txt_path = os.path.join(SAVE_DIR, "multiseed_results.txt")
with open(txt_path, "w") as f:
    f.write(tracker.summary() + "\n")
print('saved ->', res_path, '\nsaved ->', txt_path)


saved -> ./saved/multiseed_results.json 
saved -> ./saved/multiseed_results.txt


## Reload a saved model (for evaluation)

Each trained model is saved to `./saved/models/{ltn,noltn}_seed{N}.pt`.

```python
import torch
from utils import MultiKANModel
from attention_modules import AttentionKANModel
from kan import KAN

ck = torch.load('./saved/models/ltn_seed0.pt', map_location=device)
cfg = ck['config']
kan = KAN(width=cfg['KAN_WIDTH'], grid=5, k=3, seed=ck['seed'], device=device,
          auto_save=False, save_act=False)
model = AttentionKANModel(cfg['IN_FEATURES'], MultiKANModel(kan), **cfg['ATTN']).to(device)
model.load_state_dict(ck['model_state'])
model.eval()

# re-apply the saved scaler to raw features before inference:
#   X = (X_raw - ck['scaler_mean']) / ck['scaler_scale']
```

Note: `load_state_dict` restores the numeric spline weights; pykan's symbolic
lambdas are not in the state_dict (fine for numeric-mode inference).
